In [8]:
import requests
import pandas as pd
import numpy as np
import ast

In [3]:
url = "https://api.opendota.com/api/publicMatches"
print("Pinging the OpenDota API...")

Pinging the OpenDota API...


In [4]:
response = requests.get(url)

if response.status_code == 200:

    raw_data = response.json()
    
    df = pd.DataFrame(raw_data)
    
    print("Success! Here are the first 5 matches:")
    display(df.head()) 
else:
    print(f"Uh oh, something went wrong. Status code: {response.status_code}")

Success! Here are the first 5 matches:


,match_id,match_seq_num,radiant_win,start_time,duration,lobby_type,game_mode,avg_rank_tier,num_rank_tier,cluster,radiant_team,dire_team
0,8715050558,7322678192,False,1772599719,0,4,1,61,1,182,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
1,8715050366,7322678388,False,1772599699,0,4,1,33,1,227,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
2,8715049426,7322677224,False,1772599574,0,4,1,44,1,151,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
3,8715047777,7322675916,False,1772599394,0,4,1,44,1,152,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
4,8715043875,7322677373,False,1772598994,472,0,23,33,5,227,"[93, 64, 43, 84, 44]","[11, 135, 136, 13, 95]"


In [5]:
df.to_csv("dota_raw_data.csv", index=False)
print("Data saved locally to dota_raw_data.csv!")

Data saved locally to dota_raw_data.csv!


In [6]:
df = pd.read_csv("dota_raw_data.csv")

In [7]:
df['radiant_team'] =  df['radiant_team'].apply(ast.literal_eval)
df['dire_team'] = df['dire_team'].apply(ast.literal_eval)

print("Data loaded successfully! Here is what the target variable and features look like:")
display(df[['match_id', 'radiant_win', 'radiant_team', 'dire_team']].head())

Data loaded successfully! Here is what the target variable and features look like:


,match_id,radiant_win,radiant_team,dire_team
0,8715050558,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
1,8715050366,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
2,8715049426,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
3,8715047777,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
4,8715043875,False,"[93, 64, 43, 84, 44]","[11, 135, 136, 13, 95]"


In [14]:
print("Building the hero matrix... this might take a few seconds.")

matrix_rows = []

for i, row in df.iterrows():
    
    # Create empty dictionary for match
    match_features  =  {}
    
    # +1 for Radiant hero
    for hero in row['radiant_team']:
        match_features[f'hero_{hero}'] = 1
    
    # -1 for Dire hero
    for hero in row['dire_team']:
        match_features[f'hero_{hero}'] = -1
        
    matrix_rows.append(match_features)
    
hero_df = pd.DataFrame(matrix_rows).fillna(0)

final_df = pd.concat([df[['match_id', 'radiant_win']], hero_df], axis=1)

print("Matrix complete! Here is the shape of our new dataset (Rows, Columns):")
print(final_df.shape)
display(final_df.head())

Building the hero matrix... this might take a few seconds.
Matrix complete! Here is the shape of our new dataset (Rows, Columns):
(100, 126)


,match_id,radiant_win,hero_0,hero_93,hero_64,hero_43,hero_84,hero_44,hero_11,hero_135,...,hero_120,hero_137,hero_80,hero_87,hero_58,hero_65,hero_60,hero_91,hero_46,hero_66
0,8715050558,False,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,8715050366,False,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,8715049426,False,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,8715047777,False,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8715043875,False,0.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
